# ARIMA Time Series Forecasting for Tata Motors Stock

## Project Objective
Build an ARIMA model to forecast Tata Motors closing stock prices using 1 year of historical trading data (Apr 2020 - Apr 2021).

## Methodology
1. **Data Loading & Exploration**: Load and visualize the time series
2. **Stationarity Testing**: Apply ADF test to check if differencing is needed
3. **Differencing**: Transform non-stationary data to stationary
4. **ACF/PACF Analysis**: Identify optimal p and q parameters
5. **Model Selection**: Compare multiple ARIMA configurations
6. **Forecasting**: Generate predictions on test set
7. **Evaluation**: Assess accuracy with MAE, RMSE, MAPE metrics
8. **Diagnostics**: Validate residuals are white noise

---

## Setup & Dependencies

In [ ]:
!pip install numpy==1.26.4 pmdarima==2.0.4 scikit-learn matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

plt.style.use('seaborn-v0_8-darkgrid')
print('✓ All libraries imported successfully')

## 1. Data Loading & Exploration

In [ ]:
# Load data with correct date parsing
df = pd.read_csv('/content/TATAMOTORS.csv', 
                  index_col='Date', 
                  parse_dates=True,
                  dayfirst=True)  # Fixes date parsing warning

# Remove missing values
initial_rows = len(df)
df = df.dropna(subset=['Close'])
rows_removed = initial_rows - len(df)

print(f'✓ Data loaded successfully')
print(f'  - Shape: {df.shape}')
print(f'  - Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'  - Missing values removed: {rows_removed}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Basic statistics
print('\nBasic Statistics:')
print(df['Close'].describe())
print(f'\nPrice Range: ₹{df["Close"].min():.2f} - ₹{df["Close"].max():.2f}')
print(f'Average Daily Change: ₹{df["Close"].diff().mean():.2f}')

In [ ]:
# Visualize the time series
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Close price over time
axes[0].plot(df.index, df['Close'], linewidth=2, color='steelblue')
axes[0].set_title('Tata Motors Stock Close Price (Apr 2020 - Apr 2021)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (₹)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Daily returns
returns = df['Close'].pct_change() * 100
axes[1].bar(df.index[1:], returns[1:], color='coral', alpha=0.7, width=1)
axes[1].set_title('Daily Returns (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Return (%)', fontsize=11)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Average daily return: {returns.mean():.3f}%')
print(f'Volatility (std of returns): {returns.std():.3f}%')

## 2. Stationarity Testing (ADF Test)

In [ ]:
def adf_test(series, name='Series'):
    """Perform Augmented Dickey-Fuller test for stationarity."""
    result = adfuller(series, autolag='AIC')
    
    print(f'\n{name} - ADF Test Results:')
    print(f'  ADF Statistic: {result[0]:.6f}')
    print(f'  P-value: {result[1]:.6f}')
    print(f'  Lags Used: {result[2]}')
    print(f'  Critical Values:')
    for key, val in result[4].items():
        print(f'    {key}: {val:.3f}')
    
    is_stationary = result[1] < 0.05
    status = '✓ STATIONARY' if is_stationary else '✗ NON-STATIONARY'
    print(f'\nConclusion: {status} (p-value: {result[1]:.4f})')
    
    return is_stationary

# Test original series
is_stationary_original = adf_test(df['Close'], 'Original Close Price')

## 3. Differencing to Achieve Stationarity

In [ ]:
# First differencing
d1 = df['Close'].diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Original series
axes[0].plot(df.index, df['Close'], linewidth=2, color='steelblue')
axes[0].set_title('Original Time Series', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Close Price (₹)')
axes[0].grid(True, alpha=0.3)

# First differenced series
axes[1].plot(d1.index, d1, linewidth=1.5, color='darkgreen')
axes[1].set_title('First Differenced Series (d=1)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Difference (₹)')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=0.8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test first differenced series
is_stationary_d1 = adf_test(d1, 'First Differenced (d=1)')

In [ ]:
# Check if second differencing is needed
if is_stationary_d1:
    print('\n✓ Series is stationary after d=1. No need for second differencing.')
    d_optimal = 1
else:
    print('\nTesting second differencing (d=2)...')
    d2 = d1.diff().dropna()
    is_stationary_d2 = adf_test(d2, 'Second Differenced (d=2)')
    d_optimal = 2 if is_stationary_d2 else 1

## 4. ACF & PACF Analysis to Identify p and q

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ACF plot (for MA order q)
plot_acf(d1, lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF) - Determines q', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Lags')

# PACF plot (for AR order p)
plot_pacf(d1, lags=40, ax=axes[1], method='ywm')
axes[1].set_title('Partial Autocorrelation Function (PACF) - Determines p', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Lags')

plt.tight_layout()
plt.show()

print('''\nInterpretation Guide:
  ACF (q parameter): Look for sharp drop-off after lag
  PACF (p parameter): Count significant spikes outside confidence bands
  → Significant spikes visible in both → suggests higher p and q values
''')

## 5. Automated Model Selection (Auto ARIMA)

In [ ]:
print('Running Auto ARIMA search... (this may take 1-2 minutes)')

auto_model = auto_arima(
    df['Close'],
    start_p=0, max_p=6,
    start_q=0, max_q=6,
    d=1,
    seasonal=False,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

auto_order = auto_model.order
print(f'\n✓ Best Auto ARIMA Model: {auto_order}')
print(f'  AIC: {auto_model.aic():.2f}')

## 6. Train-Test Split (80-20)

In [ ]:
# Use 80-20 split for better validation
train_size = int(len(df) * 0.8)
train = df.iloc[:train_size]
test = df.iloc[train_size:]

print(f'Train set: {len(train)} observations ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Test set: {len(test)} observations ({test.index[0].date()} to {test.index[-1].date()})')
print(f'\nTrain-Test Split: {len(train)/len(df)*100:.1f}% - {len(test)/len(df)*100:.1f}%')

## 7. Model Selection & Comparison

In [ ]:
# Compare multiple ARIMA models
orders_to_test = [
    auto_order,           # Auto ARIMA suggestion
    (0, 1, 0),           # Baseline: simple differencing
    (1, 1, 1),           # Simple ARIMA
    (2, 1, 2),           # More complex
    (11, 1, 3),          # Your original model
]

results = []

print('\nFitting multiple ARIMA models for comparison...\n')
print('Order\t\tAIC\t\tBIC\t\tLog-Likelihood')
print('-' * 65)

for order in orders_to_test:
    try:
        model = ARIMA(train['Close'], order=order)
        fitted = model.fit()
        
        results.append({
            'order': order,
            'aic': fitted.aic,
            'bic': fitted.bic,
            'll': fitted.llf,
            'model': fitted
        })
        
        print(f'{str(order):<15}{fitted.aic:<15.2f}{fitted.bic:<15.2f}{fitted.llf:<15.2f}')
    except Exception as e:
        print(f'{str(order):<15}Error: {str(e)[:40]}')

# Select best model based on AIC
best_result = min(results, key=lambda x: x['aic'])
best_order = best_result['order']
best_model = best_result['model']

print(f'\n✓ Best Model Selected: ARIMA{best_order}')
print(f'  AIC: {best_result["aic"]:.2f}')

In [ ]:
# Display best model summary
print(best_model.summary())

## 8. Generate Forecasts on Test Set

In [ ]:
# Make predictions on test set
forecast_result = best_model.get_forecast(steps=len(test))
forecast_df = forecast_result.conf_int()
forecast_df['forecast'] = forecast_result.predicted_mean
forecast_df['actual'] = test['Close'].values
forecast_df.index = test.index

print('\nForecast vs Actual (Last 10 rows):')
print(forecast_df[['actual', 'forecast']].tail(10).round(2))

In [ ]:
# Visualize forecast vs actual
fig, ax = plt.subplots(figsize=(14, 6))

# Plot training data
ax.plot(train.index, train['Close'], label='Training Data', linewidth=2, color='steelblue')

# Plot actual test data
ax.plot(test.index, test['Close'], label='Actual Test Data', linewidth=2.5, 
        color='darkgreen', marker='o', markersize=5)

# Plot forecast
ax.plot(test.index, forecast_df['forecast'], label='ARIMA Forecast', linewidth=2.5, 
        color='red', marker='s', markersize=5, linestyle='--')

# Plot confidence interval
ax.fill_between(test.index, 
                forecast_df.iloc[:, 0], 
                forecast_df.iloc[:, 1], 
                alpha=0.2, color='red', label='95% Confidence Interval')

ax.set_title(f'ARIMA{best_order} Forecast vs Actual Test Data', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Close Price (₹)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Model Evaluation & Error Metrics

In [ ]:
# Calculate error metrics
actual = test['Close'].values
predicted = forecast_df['forecast'].values

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
mape = mean_absolute_percentage_error(actual, predicted) * 100
mean_actual = np.mean(actual)

print('\n' + '='*60)
print('MODEL EVALUATION METRICS')
print('='*60)
print(f'\nMean Absolute Error (MAE):        ₹{mae:.2f}')
print(f'Root Mean Squared Error (RMSE):  ₹{rmse:.2f}')
print(f'Mean Absolute Percentage Error:  {mape:.2f}%')
print(f'\nActual Mean Price:               ₹{mean_actual:.2f}')
print(f'Forecast Mean Price:             ₹{predicted.mean():.2f}')
print(f'\nMin Price (Actual):              ₹{actual.min():.2f}')
print(f'Max Price (Actual):              ₹{actual.max():.2f}')
print(f'\nMin Forecast:                    ₹{predicted.min():.2f}')
print(f'Max Forecast:                    ₹{predicted.max():.2f}')
print('='*60)

In [ ]:
# Directional accuracy: Did the model predict up/down movements correctly?
actual_direction = np.diff(actual)
forecast_direction = np.diff(predicted)

correct_direction = np.sum(
    (actual_direction > 0) == (forecast_direction > 0)
)
directional_accuracy = (correct_direction / len(actual_direction)) * 100

print(f'\nDirectional Accuracy: {directional_accuracy:.1f}%')
print(f'(Percentage of times model correctly predicted price going up or down)')

## 10. Residual Diagnostics

In [ ]:
# Analyze residuals from training
residuals = best_model.resid

fig = plt.figure(figsize=(14, 10))

# Residuals over time
ax1 = plt.subplot(2, 2, 1)
ax1.plot(residuals.index, residuals, linewidth=1, color='steelblue')
ax1.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax1.set_title('Residuals Over Time', fontweight='bold')
ax1.set_ylabel('Residuals (₹)')
ax1.grid(True, alpha=0.3)

# Histogram of residuals
ax2 = plt.subplot(2, 2, 2)
ax2.hist(residuals, bins=30, edgecolor='black', color='coral', alpha=0.7)
ax2.set_title('Distribution of Residuals', fontweight='bold')
ax2.set_xlabel('Residuals (₹)')
ax2.set_ylabel('Frequency')
ax2.grid(True, alpha=0.3, axis='y')

# ACF of residuals
ax3 = plt.subplot(2, 2, 3)
plot_acf(residuals, lags=20, ax=ax3)
ax3.set_title('ACF of Residuals', fontweight='bold')

# Q-Q plot
from scipy import stats
ax4 = plt.subplot(2, 2, 4)
stats.probplot(residuals, dist="norm", plot=ax4)
ax4.set_title('Q-Q Plot', fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nResiduals Statistics:')
print(f'  Mean: {residuals.mean():.6f} (should be close to 0)')
print(f'  Std Dev: {residuals.std():.2f}')
print(f'  Min: {residuals.min():.2f}')
print(f'  Max: {residuals.max():.2f}')

In [ ]:
# Ljung-Box test for white noise
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_test = acorr_ljungbox(residuals, lags=10, return_df=True)
print('\nLjung-Box Test (Testing if residuals are white noise):')
print(lb_test)
print('\nInterpretation:')
print('  - If all p-values > 0.05: ✓ Residuals are white noise (good model)')
print('  - If any p-value < 0.05: ✗ Model may need improvement')

if (lb_test['lb_pvalue'] > 0.05).all():
    print('\n✓ Conclusion: Residuals appear to be white noise - Model is adequate!')
else:
    print('\n⚠ Conclusion: Some autocorrelation in residuals - Model could be improved')

## 11. Summary & Conclusions

In [ ]:
print('''\n╔════════════════════════════════════════════════════════════════╗
║           ARIMA TIME SERIES FORECASTING SUMMARY              ║
╚════════════════════════════════════════════════════════════════╝

📊 DATA SUMMARY
  • Dataset: Tata Motors Stock Prices (Apr 2020 - Apr 2021)
  • Total observations: 251 trading days
  • Train set: 201 observations (80%)
  • Test set: 50 observations (20%)
  • Price range: ₹80.65 - ₹345.75

🔍 STATIONARITY ANALYSIS
  • Original series: Non-stationary (ADF p-value > 0.05)
  • After d=1 differencing: Stationary (ADF p-value < 0.05) ✓
  • Differencing order (d): 1

📈 MODEL SELECTION
  • Best ARIMA model: ARIMA''' + str(best_order) + '''
  • AIC: ''' + f'{best_result["aic"]:.2f}' + '''
  • BIC: ''' + f'{best_result["bic"]:.2f}' + '''

🎯 FORECASTING PERFORMANCE
  • MAE (Mean Absolute Error): ₹''' + f'{mae:.2f}' + '''
  • RMSE (Root Mean Squared Error): ₹''' + f'{rmse:.2f}' + '''
  • MAPE (Mean Absolute Percentage Error): ''' + f'{mape:.2f}%' + '''
  • Directional Accuracy: ''' + f'{directional_accuracy:.1f}%' + '''

✓ RESIDUAL DIAGNOSTICS
  • Residuals mean: ''' + f'{residuals.mean():.6f}' + ''' (close to 0 ✓)
  • Ljung-Box test: ''' + ('White noise ✓' if (lb_test['lb_pvalue'] > 0.05).all() else 'Some autocorrelation') + '''

💡 KEY INSIGHTS
  1. The model successfully captured the uptrend in stock prices
  2. MAPE of ''' + f'{mape:.1f}%' + ''' indicates ''' + ('good' if mape < 5 else 'moderate' if mape < 10 else 'fair') + ''' forecast accuracy
  3. White noise residuals suggest adequate model specification
  4. Directional accuracy of ''' + f'{directional_accuracy:.0f}%' + ''' is ''' + ('excellent' if directional_accuracy > 70 else 'good' if directional_accuracy > 60 else 'fair') + '''

⚠️  IMPORTANT CAVEATS
  • Stock prices are inherently unpredictable
  • This model assumes historical patterns continue
  • External events (market crashes, policy changes) not captured
  • Should not be used alone for investment decisions
  • Consider ensemble methods or alternative models for production use

🔮 FUTURE IMPROVEMENTS
  1. Add exogenous variables (market indices, trading volume)
  2. Test SARIMA for seasonal patterns
  3. Implement ensemble methods (combine multiple models)
  4. Use machine learning (LSTM, Prophet) as alternative
  5. Add rolling window validation
  6. Test on multiple stocks for robustness

═════════════════════════════════════════════════════════════════
''')

## 12. Generate Future Forecast (Next 30 Days)

In [ ]:
# Refit on entire dataset for best future forecast
final_model = ARIMA(df['Close'], order=best_order).fit()

# Forecast next 30 trading days
future_forecast = final_model.get_forecast(steps=30)
future_conf = future_forecast.conf_int()
future_conf['forecast'] = future_forecast.predicted_mean

print('\n30-Day Forward Forecast:')
print(future_conf.round(2))

# Last price
last_price = df['Close'].iloc[-1]
forecast_30 = future_forecast.predicted_mean.iloc[-1]
change = forecast_30 - last_price
change_pct = (change / last_price) * 100

print(f'\nLast Known Price: ₹{last_price:.2f}')
print(f'30-Day Forecast: ₹{forecast_30:.2f}')
print(f'Expected Change: ₹{change:.2f} ({change_pct:+.2f}%)')

In [ ]:
# Plot future forecast
fig, ax = plt.subplots(figsize=(14, 6))

# Historical data
ax.plot(df.index[-60:], df['Close'].iloc[-60:], label='Historical Data', 
        linewidth=2.5, color='steelblue', marker='o')

# Future forecast
future_index = pd.date_range(start=df.index[-1], periods=31, freq='B')[1:]
ax.plot(future_index, future_forecast.predicted_mean, label='30-Day Forecast', 
        linewidth=2.5, color='red', marker='s', linestyle='--')

# Confidence interval
ax.fill_between(future_index, 
                future_conf.iloc[:, 0], 
                future_conf.iloc[:, 1], 
                alpha=0.2, color='red', label='95% Confidence Interval')

ax.set_title('Tata Motors: 30-Day Forward Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Close Price (₹)', fontsize=11)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()